In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install biopython

In [ ]:
!pip install scipy --target=$nb_path

In [ ]:
!pip install matplotlib ipywidgets --target=$nb_path

In [ ]:
!apt-get install -y python3-pymol

In [ ]:
!pip install trimesh

In [ ]:
!pip install pyglet

In [ ]:
!pip install pyvista

In [ ]:
!pip install pywavefront

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
from Bio import PDB
from google.colab import files

In [ ]:
sys.path.append('/content/drive/MyDrive/antigen/')
uploaded = files.upload()
!pip install pymesh2-0.3.1-cp310-cp310-linux_x86_64.whl --upgrade

In [ ]:
import pymesh
from Bio.PDB import PDBParser, PDBIO
from scipy.spatial import Delaunay
import trimesh
import itertools
from scipy.spatial import ConvexHull
import pyvista as pv
from pymol import cmd
import matplotlib.pyplot as plt
from collections import defaultdict
import pywavefront

In [ ]:
local_path = 'drive/MyDrive/antigen/'
filename = '1a2y_B_A_C.obj'

In [ ]:
mesh = pywavefront.Wavefront(local_path+filename, collect_faces=True)
vertices = mesh.vertices

In [ ]:
def calculate_curvature(pdb_file_path, n_neighbors, distance_threshold = 5.0):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('protein', pdb_file_path)
    model = structure[0]
    chain = model['C']
    atoms = [atom for atom in chain.get_atoms()]

    # Remove HOH Atom
    # atoms = [atom for atom in chain.get_atoms() if atom.get_parent().resname != 'HOH']

    coords = np.array([atom.coord for atom in atoms])
    curvatures = [None] * len(atoms)

    # temp_counter = 0
    for idx, atom in enumerate(atoms):
        atom_coords = np.array(coords[idx])
        dist_coords = coords - atom_coords

        # Calculate distances and get indices of nearest neighbors
        dists = np.linalg.norm(dist_coords, axis=1)
        sorted_indices = np.argsort(dists)

        # Decided by threshold, not n_neighbors
        nearest_neighbors_idx = sorted_indices[np.logical_and(dists[sorted_indices] > 0, dists[sorted_indices] < distance_threshold)]


        # # Exclude the atom itself by starting from index 1
        # within_threshold_indices = sorted_indices[np.logical_and(dists[sorted_indices] > 0, dists[sorted_indices] <= distance_threshold)]
        # # If there are fewer than n_neighbors within the distance threshold, increase the threshold and repeat
        # while len(within_threshold_indices) < n_neighbors:
        #     distance_threshold += 0.5
        #     within_threshold_indices = sorted_indices[np.logical_and(dists[sorted_indices] > 0, dists[sorted_indices] <= distance_threshold)]
        # nearest_neighbors_idx = within_threshold_indices[:n_neighbors]


        # Create vertices and faces
        vertices = dist_coords[np.concatenate([[idx], nearest_neighbors_idx]), :]

        cloud = pv.PolyData(vertices)
        triangulated = cloud.delaunay_2d()
        shell = triangulated.extract_geometry()

        extracted_vertices = np.array(shell.points)
        extracted_faces = np.array(shell.faces).reshape(-1, 4)[:, 1:]
        extracted_faces = np.array([face for face in extracted_faces if 0 in face])

        # Connect faces around vertex
        vertex_count = defaultdict(int)
        for face in extracted_faces:
          if 0 in face:
            vertices = face[face != 0]
            for vertex in vertices:
              vertex_count[vertex] += 1
        single_occurrences = [vertex for vertex, count in vertex_count.items() if count == 1]

        if len(single_occurrences) == 2:
          new_face = single_occurrences + [0]
          extracted_faces = np.vstack([extracted_faces, new_face])

        mesh = pymesh.form_mesh(extracted_vertices, extracted_faces)

        # Remove isolated points
        mesh, info = pymesh.remove_isolated_vertices(mesh)

        # Taubin Smoothing (0.3, -1.06, 30)
        lambda_factor = 0.3
        mu_factor = -1.06 * lambda_factor
        vertices_new = mesh.vertices.copy()
        for _ in range(30):
          laplacian = np.zeros_like(vertices_new)
          for face in mesh.faces:
            for i, vertex in enumerate(face):
              adj_vertices = face[np.arange(len(face)) != i]
              laplacian[vertex] += vertices_new[adj_vertices].mean(axis=0)

          movement = laplacian - vertices_new
          vertices_new += lambda_factor * movement
          vertices_new += mu_factor * movement

          mesh = pymesh.form_mesh(vertices_new, mesh.faces)
          mesh, info = pymesh.remove_isolated_vertices(mesh)

        mesh.add_attribute("vertex_mean_curvature")
        H = mesh.get_attribute("vertex_mean_curvature")
        mean_H = H[0]

        mesh.add_attribute("vertex_gaussian_curvature")
        K = mesh.get_attribute("vertex_gaussian_curvature")
        K = K[0]

        elem = np.square(mean_H) - K
        if elem < 0:
          elem = 1e-8
        elem = np.array(elem)

        k1 = mean_H + np.sqrt(elem)
        k2 = mean_H - np.sqrt(elem)

        si = (k1 + k2) / (k1 - k2)
        si = np.arctan(si) * (2 / np.pi)

        curvatures[idx] = si

    return curvatures

def select_and_update(structure, curvatures):
    for model in structure:
        for chain in model:
            if chain.id == 'C':
                for i, residue in enumerate(chain):
                    for j, atom in enumerate(residue):
                        if i * len(residue) + j < len(curvatures):
                            curvature_value = curvatures[i * len(residue) + j]
                            if curvature_value is not None:
                                atom.set_bfactor(curvature_value)

In [ ]:
# Initialize a PDB parser
parser = PDBParser(QUIET=True)
filename = '1a2y_B_A_C.pdb'
# Load the structure
pdb_file_path = local_path + filename
structure = parser.get_structure('1a2y_B_A_C', pdb_file_path)

# Calculate curvatures
n_neighbors_list = 10
distance_threshold = [5, 6, 7]
for d in distance_threshold:
    curvatures = calculate_curvature(pdb_file_path, n_neighbors_list, d)

    # Create a new PDBIO object for saving the modified structure
    io = PDBIO()

    # Save the updated chain C structure
    output_path = f'Taubin_1a2y_B_A_C_threshold{d}__curvature.pdb'
    io.set_structure(structure)
    select_and_update(structure, curvatures)
    io.save(output_path)

In [ ]:
def delete_ply_files_in_current_directory():
    current_directory = os.getcwd()
    for filename in os.listdir(current_directory):
        if filename.endswith('.pdb'):
            os.remove(os.path.join(current_directory, filename))
            print(f"Deleted: {filename}")
delete_ply_files_in_current_directory()

In [ ]:
#@title Read the data frame from csv file
df = pd.read_csv(os.path.join(local_path, '1a2y_B_A_C_info.tsv'),delimiter='\t')

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
df.head(20)

In [ ]:
# Filter out surface residues
surface_residues = df[df['SaSaRate'] > 0.2]

# Initialize an empty list to store curvature
curvatures = [None] * len(df)

distance_threshold = 5

# Loop over each surface residue to find its 3 nearest neighbors
for idx, residue in surface_residues.iterrows():
    coords = df[['SCx', 'SCy', 'SCz']].values.astype(float)
    residue_coords = np.array(residue[['SCx', 'SCy', 'SCz']].values, dtype=float)
    dists = np.linalg.norm(coords - residue_coords, axis=1)
    sorted_indices = np.argsort(dists)

    # nearest_neighbors_idx = sorted_indices[np.logical_and(dists[sorted_indices] > 0, dists[sorted_indices] < 10)]

    # Exclude the atom itself by starting from index 1
    within_threshold_indices = sorted_indices[np.logical_and(dists[sorted_indices] > 0, dists[sorted_indices] <= distance_threshold)]
    # If there are fewer than n_neighbors within the distance threshold, increase the threshold and repeat
    while len(within_threshold_indices) < 4:
        distance_threshold += 0.25
        within_threshold_indices = sorted_indices[np.logical_and(dists[sorted_indices] > 0, dists[sorted_indices] <= distance_threshold)]
    nearest_neighbors_idx = within_threshold_indices


    # nearest_neighbors_idx = np.argsort(dists)[1:4]  # Exclude itself

    # Create vertices (coordinates of the residue and its neighbors)
    vertices = coords[np.concatenate([[idx], nearest_neighbors_idx]), :]

    # Create faces (assuming the points form a tetrahedron)
    cloud = pv.PolyData(vertices)
    triangulated = cloud.delaunay_2d()
    shell = triangulated.extract_geometry()

    extracted_vertices = np.array(shell.points)
    extracted_faces = np.array(shell.faces).reshape(-1, 4)[:, 1:]
    extracted_faces = np.array([face for face in extracted_faces if 0 in face])

    vertex_count = defaultdict(int)
    for face in extracted_faces:
        if 0 in face:
            vertices = face[face != 0]
            for vertex in vertices:
                vertex_count[vertex] += 1
    single_occurrences = [vertex for vertex, count in vertex_count.items() if count == 1]


    if len(single_occurrences) == 2:
      new_face = single_occurrences + [0]
      extracted_faces = np.vstack([extracted_faces, new_face])

    # print(extracted_faces)

    # Create mesh in PyMesh
    mesh = pymesh.form_mesh(extracted_vertices, extracted_faces)
    mesh, info = pymesh.remove_isolated_vertices(mesh)


    lambda_factor = 0.3
    mu_factor = -1.06 * lambda_factor
    vertices_new = mesh.vertices.copy()
    for _ in range(30):
      laplacian = np.zeros_like(vertices_new)
      for face in mesh.faces:
        for i, vertex in enumerate(face):
          adj_vertices = face[np.arange(len(face)) != i]
          laplacian[vertex] += vertices_new[adj_vertices].mean(axis=0)

      movement = laplacian - vertices_new
      vertices_new += lambda_factor * movement
      vertices_new += mu_factor * movement

      # Recreate the mesh after smoothing
      mesh = pymesh.form_mesh(vertices_new, mesh.faces)
      mesh, info = pymesh.remove_isolated_vertices(mesh)

    # Calculate Gaussian curvature
    mesh.add_attribute("vertex_gaussian_curvature")
    curvature = mesh.get_attribute("vertex_gaussian_curvature")

    mesh.add_attribute("vertex_mean_curvature")
    H = mesh.get_attribute("vertex_mean_curvature")
    mean_H = H[0]

    mesh.add_attribute("vertex_gaussian_curvature")
    K = mesh.get_attribute("vertex_gaussian_curvature")
    K = K[0]

    elem = np.square(mean_H) - K
    if elem < 0:
      elem = 1e-8
    elem = np.array(elem)

    k1 = mean_H + np.sqrt(elem)
    k2 = mean_H - np.sqrt(elem)

    si = (k1 + k2) / (k1 - k2)
    si = np.arctan(si) * (2 / np.pi)

    curvatures[idx] = si

# Add the curvature column to the DataFrame
df['curvatures'] = curvatures

In [ ]:
# Initialize a PDB parser
parser = PDB.PDBParser(QUIET=True)

structure = parser.get_structure('1a2y_B_A_C', local_path + '1a2y_B_A_C.pdb')

# Create a dictionary with ResID as key and curvature as value
curvature_dict = dict(zip(df['ResID'], df['curvatures']))

# Iterate through residues and update b-factors
for model in structure:
    for chain in model:
        for residue in chain:
            res_id = residue.id[1]  # Get residue number
            if res_id in curvature_dict:
                curvature_value = curvature_dict[res_id]
                if not pd.isna(curvature_value):  # Check if curvature value is NaN
                    for atom in residue:
                        atom.set_bfactor(curvature_value)

# Save the modified structure to a new file
io = PDB.PDBIO()
io.set_structure(structure)
io.save('1a2y_B_A_C_surface_curvature_06.pdb')